# Load the MNIST dataset

https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data

In [1]:
import tensorflow as tf
import numpy as np
import cv2

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
assert x_train.shape == (60000, 28, 28)
assert x_test.shape == (10000, 28, 28)
assert y_train.shape == (60000,)
assert y_test.shape == (10000,)

In [32]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import cv2

def load_kmnist():
    # Load KMNIST dataset
    dataset, info = tfds.load('kmnist', split=['train', 'test'], shuffle_files=True, as_supervised=True, with_info=True)
    
    # Split the dataset into train and test
    train_dataset, test_dataset = dataset
    
    # Extract images and labels
#     x_train, y_train = [], []
#     x_test, y_test = [], []
    x_train = np.zeros([60000, 28, 28],np.int)
    y_train = np.zeros(60000,np.int)
    x_test = np.zeros([10000, 28, 28],np.int)
    y_test = np.zeros(10000,np.int)
    for ind, (image, label) in enumerate(tfds.as_numpy(train_dataset)):
        x_train[ind,:,:]= image.squeeze()
        y_train[ind]=label
    for ind,(image, label) in enumerate(tfds.as_numpy(test_dataset)):
        x_test[ind,:,:]= image.squeeze()
        y_test[ind]=label
    
    return x_train, y_train, x_test, y_test

# Test the function
x_train, y_train, x_test, y_test = load_kmnist()
assert x_train.shape == (60000, 28, 28)
assert x_test.shape == (10000, 28, 28)
assert y_train.shape == (60000,)
assert y_test.shape == (10000,)
# print("Number of training samples:", len(x_train))
# print("Number of test samples:", len(x_test))

In [2]:
x_traintest = np.vstack((x_train,x_test))
y_traintest = np.hstack((y_train,y_test))
# x_traintest = x_train
# y_traintest = y_train

traintest_randperm = np.random.permutation(y_traintest.shape[0])
x_traintest = x_traintest[traintest_randperm]
y_traintest = y_traintest[traintest_randperm]

#x_traintest = x_traintest.repeat(3, axis=0)
#y_traintest = y_traintest.repeat(3)

print(x_traintest.shape)
print(y_traintest.shape)

(70000, 28, 28)
(70000,)


In [3]:
## insert 'vernier' samples into the sequence
x_traintest_lst = list(x_traintest)
y_traintest_lst = list(y_traintest)

empty_image = np.zeros((28,28), np.int)

vernier_every_n_sym = 99
vernier_to_add = len(y_traintest_lst)//vernier_every_n_sym

vernier_step = [13,14,15,16,17,18,19,21,22,23,24,25,26,27]
vernier_step = np.tile(vernier_step, vernier_to_add//len(vernier_step)+1)
vernier_step = vernier_step[:vernier_to_add+1]

vernier_inexes = np.arange(0, len(y_traintest_lst) ,vernier_every_n_sym)

for index, ver_step in zip(reversed(vernier_inexes), reversed(vernier_step)):
#     print(index, ver_step)
    x_traintest_lst.insert(index, empty_image)
    y_traintest_lst.insert(index, ver_step)

x_traintest_ = np.array(x_traintest_lst)
y_traintest_ = np.array(y_traintest_lst)

print(x_traintest.shape)
print(x_traintest_.shape)
print(y_traintest.shape)
print(y_traintest_.shape)

x_traintest = x_traintest_
y_traintest = y_traintest_

print(len(vernier_step))
print(len(vernier_inexes))

(70000, 28, 28)
(70708, 28, 28)
(70000,)
(70708,)
708
708


In [42]:
# Servo class

import time
import maestro

class Servo(maestro.Controller):
    def __init__(self):
        maestro.Controller.__init__(self, ttyStr = '/dev/ttyACM2')
        self.a_com_period_s=0.005
        self.a_speed=0.0005
        
    def pos_abs(self,axis=0, pos_a=6000,vel=1, acc=8):
#         motion_t=3 # add calc
        servo.setAccel(axis,acc)
        servo.setSpeed(axis,vel)
#         time.sleep(1)
        servo.setTarget(axis,pos_a)   
#         time.sleep(motion_t)
    
    def pos_rel(self,axis=0, pos_r=0,vel=1, acc=8):
        pos_now=servo.getPosition(axis)
#         motion_t=3 # add calc
        servo.setAccel(axis,acc)
        servo.setSpeed(axis,vel)
#         time.sleep(1)
        servo.setTarget(axis,pos_now+pos_r)   
#         time.sleep(motion_t)
    
    def pos_abs_ds(self,axis=0, pos_a=6000):
        servo.setTarget(axis,pos_a)   
    
    def pos_rel_ds(self,axis=0, pos_r=0):
        pos_now=servo.getPosition(axis)
        servo.setTarget(axis,pos_now+pos_r)   

if __name__ == "__main__":
    servo = Servo()
    time.sleep(1)
#     servo.pos_abs(axis=0, pos_a=6000,vel=1, acc=8)
#     servo.pos_abs(axis=1, pos_a=7900,vel=1, acc=8)
##    servo.pos_abs(axis=0, pos_a=6000,vel=5, acc=15)
##    servo.pos_abs(axis=1, pos_a=7900,vel=5, acc=15)
    servo.pos_abs(axis=0, pos_a=6000,vel=4, acc=0)
    servo.pos_abs(axis=1, pos_a=7620,vel=4, acc=0)
    time.sleep(1)
    print(servo.getPosition(0)) #get the current position of servo
    print(servo.getPosition(1)) #get the current position of servo
#    servo.close()

6000
7620


# Add landmark lines to image

In [6]:
# run this cell

import cv2

lcd_res= (1080,1920) # (1024,1280)
blank_fr=np.zeros((lcd_res[0],lcd_res[1]))
image=blank_fr
im_shape=image.shape
print(im_shape)
image_lines=image
row_white=np.ones(im_shape[1])*255
col_white=np.ones(im_shape[0])*255

row_white_coord=700
col_white_coord=800
line_width=1
#for j in range(line_width):
#    image_lines[row_white_coord+j,:]=row_white
#    image_lines[:,col_white_coord+j]=col_white

cv2.namedWindow ('screen_greed', cv2.WINDOW_NORMAL)
cv2.setWindowProperty ('screen_greed', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.imshow('screen_greed', image)
cv2.waitKey(2000)
cv2.destroyAllWindows()

(1080, 1920)


In [4]:
# run this cell

import cv2

lcd_res= (1080,1920) # (1024,1280)
blank_fr=np.zeros((lcd_res[0],lcd_res[1]))
image=blank_fr
im_shape=image.shape
print(im_shape)
image_lines=image
row_white=np.ones(im_shape[1])*255
col_white=np.ones(im_shape[0])*255

row_white_coord=lcd_res[0]//2+300
col_white_coord=lcd_res[1]//2
#line_width=1
line_width=0
for j in range(line_width):
    image_lines[row_white_coord+j,:]=row_white
    image_lines[:,col_white_coord+j]=col_white
    
#y=(lcd_res[0]//2+400)-tar_h_pix//2
#x=lcd_res[1]//2+310-tar_w_pix//2
y=(lcd_res[0]//2)+100-tar_h_pix//2
x=lcd_res[1]//2+100-tar_w_pix//2
image[y:y+tar_h_pix,x:x+tar_w_pix]=x_train[0,:,:]/255.0

cv2.namedWindow ('screen_greed', cv2.WINDOW_NORMAL)
cv2.setWindowProperty ('screen_greed', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.imshow('screen_greed', image)
cv2.waitKey(10000)
cv2.destroyAllWindows()

(1080, 1920)


NameError: name 'tar_h_pix' is not defined

In [26]:
def generate_v_vernier(edge_position=500, step_position=640 , vernier_shift = 1, contrast = 1.0, edge_half_height=0 ):
    blank_fr=np.zeros((lcd_res[0],lcd_res[1]))
    image=blank_fr
    
    if edge_half_height != 0:
        image[step_position-edge_half_height:step_position,:edge_position] = contrast
        image[step_position:step_position+edge_half_height,:edge_position+vernier_shift] = contrast
    else:
        image[:step_position,:edge_position] = contrast
        image[step_position:,:edge_position+vernier_shift] = contrast
        
    
    return image

tar_w_pix=28
tar_h_pix=28

cv2.namedWindow ('screen_greed', cv2.WINDOW_NORMAL)
cv2.setWindowProperty ('screen_greed', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
#cv2.imshow('screen_greed', generate_v_edge(int(lcd_res[1]/2), contrast=1.0))

y=(lcd_res[0]//2+100)-tar_h_pix//2
x=lcd_res[1]//2+250-tar_w_pix//2
# cv2.imshow('screen_greed', generate_v_vernier(edge_position=int(lcd_res[1]/2+120), step_position=int(lcd_res[0]/2+250), contrast=1.0, vernier_shift = -5, edge_half_height=12))
cv2.imshow('screen_greed', generate_v_vernier(edge_position=int(lcd_res[1]/2+60), step_position=int(lcd_res[0]/2+400), contrast=1.0, vernier_shift = -5, edge_half_height=12))
cv2.waitKey(20000)

cv2.destroyAllWindows()

In [6]:
def bacground_grey_level_transform(image, bg_grey_level = 0.05):
    modified_image = np.zeros(image.shape)
    modified_image = image*(1.0-bg_grey_level)+bg_grey_level
    return modified_image

In [48]:
import numpy as np
import cv2
import time
from dv import NetworkTriggerInput
from dv import Control
import pickle

lcd_res= (1080,1920)
tar_w_pix=28
tar_h_pix=28
blank_fr=np.zeros((lcd_res[0],lcd_res[1]))
image = blank_fr

BG_GREY_LEVEL = 0.2

# --------input---------------
sym_ind_start=0
sym_ind_end= 70708 # e.g. sym_ind_end = 59999
# sym_ind_end= 50# e.g. sym_ind_end = 59999
sym_interval=1000# e.g. sym_interval=1000 sym_interval defines the number of recorded symbols per .aeda4 file
maestro_full_interval_ms=2500 # maestro full interval for a single symbol [ms]

# Change .aedat4 file prefix
sym_interval=np.min([sym_ind_end-sym_ind_start+1,sym_interval])
sym_ind_start_str=str(sym_ind_start)
sym_ind_end_str=str(sym_ind_start+sym_interval-1)
rec_file_prefix='frames_DAVIS240C_'+sym_ind_start_str+'_'+sym_ind_end_str

#/saving ds labels to pickle file
with open('./dv_ds_aedat/mnist_ds_1/ds_labels.pkl', 'wb') as flables:
     pickle.dump(y_traintest[:sym_ind_end+1], flables)
        #/saving ds labels to pickle file
with open('./dv_ds_aedat/mnist_ds_1/ds_samples_indexes.pkl', 'wb') as findex:
     pickle.dump(traintest_randperm[:sym_ind_end+1], findex)

# -------------config dvs camera parameters--------------------------------------
try:
    ctrl = Control(address='127.0.0.1', port=4040) # change to default_port=4040
    ctrl.put('/mainloop/Recorder/', 'timeout', 'long',2500000)
    print(ctrl.get('/mainloop/Recorder/', 'timeout', 'long'))
    ctrl.put('/mainloop/Recorder/', 'prefix', 'string','frames_DAVIS240C_')
    print(ctrl.get('/mainloop/Recorder/', 'prefix', 'string'))
    rec_file_dir=ctrl.get('/mainloop/Recorder/', 'directory', 'string')
    print(rec_file_dir)
    ctrl.put('/mainloop/Recorder/', 'running', 'bool', False)
    print(ctrl.get('/mainloop/Recorder/', 'running', 'bool'))
except:
    print('could not config DAVIS device')

y_shift = 355
x_shift = 130
# y=(lcd_res[0]//2+90)-tar_h_pix//2
# x=(lcd_res[1]//2+332)-tar_w_pix//2
# y=(lcd_res[0]//2+50)-tar_h_pix//2
# x=lcd_res[1]//2+75-tar_w_pix//2
# y=(lcd_res[0]//2+185)-tar_h_pix//2
# x=lcd_res[1]//2+355-tar_w_pix//2
# y=(lcd_res[0]//2+255)-tar_h_pix//2
# x=lcd_res[1]//2+235-tar_w_pix//2
y=(lcd_res[0]//2+y_shift)-tar_h_pix//2
x=lcd_res[1]//2+x_shift-tar_w_pix//2

cv2.namedWindow ('screen', cv2.WINDOW_NORMAL)
cv2.setWindowProperty ('screen', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.imshow('screen', image)
cv2.waitKey(20)
cv2.imshow('screen', image)
cv2.waitKey(20)

sym_interval=np.min([sym_ind_end-sym_ind_start,sym_interval])
sym_ind_start_str=str(sym_ind_start)
sym_ind_end_str=str(sym_ind_start+sym_interval)

#start_aedat_acquisition()
ctrl.put('/mainloop/Recorder/', 'prefix', 'string', rec_file_prefix)
print(ctrl.get('/mainloop/Recorder/', 'prefix', 'string'))
ctrl.put('/mainloop/Recorder/', 'running', 'bool', True)
#time.sleep(0.050)

sym_infile = 0
for sym_index in range(sym_ind_start, sym_ind_end):
    
#     image[y:y+tar_h_pix,x:x+tar_w_pix]=x_traintest[sym_index,:,:]/255.0
#     cv2.imshow('screen', image)
#     cv2.waitKey(20)
#     time.sleep(0.050)
     # Next symbol setup
    if y_traintest[sym_index]>10:
        #vernier_sym = True
        vernier_step = y_traintest[sym_index]-20
#                 image1=generate_v_vernier(edge_position=int(lcd_res[1]/2+250), step_position=int(lcd_res[0]/2+100), contrast=1.0, vernier_shift = vernier_step)
#                 image1=generate_v_vernier(edge_position=int(lcd_res[1]/2+75), step_position=int(lcd_res[0]/2+50), contrast=1.0, vernier_shift = vernier_step)
        image1=generate_v_vernier(edge_position=int(lcd_res[1]/2+x_shift), step_position=int(lcd_res[0]/2+y_shift), contrast=1.0, vernier_shift = vernier_step, edge_half_height=14)
        image1 = bacground_grey_level_transform(image1, BG_GREY_LEVEL)
        cv2.imshow('screen', image1)
    else:
        image[y:y+tar_h_pix,x:x+tar_w_pix]=x_traintest[sym_index,:,:]/255.0
        image1 = bacground_grey_level_transform(image, BG_GREY_LEVEL)
        cv2.imshow('screen', image1)
    
    cv2.waitKey(20)
    time.sleep(0.050)
    
    servo.setTarget(chan=6, target=7000)#raizing edge-recorded by davis
    
    time.sleep(0.100)
    
    servo.setTarget(chan=6, target=5000)#falling edge-recorded by davis
        
    #time.sleep(0.375)
    
    sym_infile += 1
    if sym_infile == sym_interval:
        sym_infile = 0
        #stop_aedat_acquisition()
        ctrl.put('/mainloop/Recorder/', 'running', 'bool', False)
        #print('file closed')
        #time.sleep(0.05)
        
        # Change .aedat4 file prefix
        sym_ind_start_str=str(sym_index+1)
        if sym_index+sym_interval<sym_ind_end:
            sym_ind_end_str=str(sym_index+1+sym_interval)
        else:
            sym_ind_end_str=str(sym_ind_end)
        
        rec_file_prefix='frames_DAVIS240C_'+sym_ind_start_str+'_'+sym_ind_end_str
        ctrl.put('/mainloop/Recorder/', 'prefix', 'string', rec_file_prefix)
        ctrl.put('/mainloop/Recorder/', 'running', 'bool', True)
        #print('file opened')
        #start_aedat_acquisition()
        #time.sleep(0.05)
        #if ctrl.get('/mainloop/capture/', 'running', 'bool')==False:
        #    print('dvs128 capture was halted, '+str(sym_index+1))
        #    ctrl.put('/mainloop/capture/', 'running', 'bool', True)
        
cv2.destroyAllWindows()
ctrl.put('/mainloop/Recorder/', 'running', 'bool', False)
servo.setTarget(0,0)
servo.setTarget(1,0)        

2500000
frames_DAVIS240C_
/home/syclopx/Workspace/jupyter_datasets/dv_ds_aedat/mnist_ds_1
False
frames_DAVIS240C_0_999


In [28]:
cv2.destroyAllWindows()
#ctrl.put('/mainloop/Recorder/', 'running', 'bool', False)
servo.setTarget(0,0)
servo.setTarget(1,0)    